
# 6. Recommendation Systems

## 6.1 Association Rule Learning

### 6.1.1 Data Preparation

We will analyze products purchased together in shopping baskets.

Our dataset contains transaction and product information.

We will:

- Select valid retail products.
- Remove non-product transaction categories.
- Create unique basket-product relationships.
- Split the data by time.
- Prepare transactions for association rule mining.

Training period: Days 1–621

Test period: Days 622–711

We will use training data to discover product relationships.


In [108]:
from pathlib import Path
import sys
from sqlalchemy import text
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import csr_matrix, save_npz
from mlxtend.frequent_patterns import fpgrowth, association_rules
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import TruncatedSVD, NMF

from functools import lru_cache
from collections import defaultdict

import joblib

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.database import get_engine

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)

engine = get_engine()

In [39]:
# Define training and test periods
TRAIN_END_DAY = 621
TEST_END_DAY = 711

In [40]:
query = text("""
    SELECT DISTINCT
        t.household_key,
        t.BASKET_ID,
        t.`DAY` AS basket_day,
        t.PRODUCT_ID

    FROM raw_transaction_data AS t
    
    INNER JOIN raw_product AS p
        ON t.PRODUCT_ID = p.PRODUCT_ID
    
    WHERE t.`DAY` BETWEEN 1 AND :test_end

        AND t.SALES_VALUE > 0
        AND t.QUANTITY > 0

        AND p.DEPARTMENT IS NOT NULL
        AND TRIM(p.DEPARTMENT) <> ''

        AND p.DEPARTMENT NOT IN (
            'KIOSK-GAS',
            'MISC SALES TRAN',
            'COUP/STR & MFG'
        )

        AND COALESCE(p.COMMODITY_DESC, '') NOT IN (
            'COUPON/MISC ITEMS',
            'FUEL'
        )

""")

with engine.connect() as connection:
    basket_items = pd.read_sql_query(
        query,
        con=connection,
        params={"test_end": TEST_END_DAY},
    )

# Validate the data
assert basket_items.notna().all().all()

assert not basket_items.duplicated(
    subset=["BASKET_ID", "PRODUCT_ID"]
).any()

assert (
    basket_items.groupby("BASKET_ID")[
        "household_key"
    ].nunique().max() == 1
)

assert (
    basket_items.groupby("BASKET_ID")[
        "basket_day"
    ].nunique().max() == 1
)

print("Basket-Product Dataset:", basket_items.shape)

display(basket_items.head(10))

Basket-Product Dataset: (2546956, 4)


,household_key,BASKET_ID,basket_day,PRODUCT_ID
0,1,27601281299,51,825123
1,1,27601281299,51,831447
2,1,27601281299,51,840361
3,1,27601281299,51,845307
4,1,27601281299,51,852014
5,1,27601281299,51,854982
6,1,27601281299,51,856942
7,1,27601281299,51,912676
8,1,27601281299,51,940947
9,1,27601281299,51,945805


In [41]:
# Split data by purchase day
train_items = basket_items.loc[
    basket_items["basket_day"] <= TRAIN_END_DAY
].copy()

test_items = basket_items.loc[
    basket_items["basket_day"] > TRAIN_END_DAY
].copy()

# Summarize the datasets
split_summary = pd.DataFrame({
    "dataset": ["Training", "Test"],
    "basket_product_rows": [
        len(train_items),
        len(test_items),
    ],
    "unique_baskets": [
        train_items["BASKET_ID"].nunique(),
        test_items["BASKET_ID"].nunique(),
    ],
    "unique_products": [
        train_items["PRODUCT_ID"].nunique(),
        test_items["PRODUCT_ID"].nunique(),
    ],
    "unique_households": [
        train_items["household_key"].nunique(),
        test_items["household_key"].nunique(),
    ],
})

display(split_summary)

# Basket size distribution
basket_sizes = (
    train_items
    .groupby("BASKET_ID")
    .size()
    .rename("unique_products")
)

print("\nTRAINING BASKET SIZE SUMMARY")

display(
    basket_sizes.describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    ).round(2)
)

,dataset,basket_product_rows,unique_baskets,unique_products,unique_households
0,Training,2180336,217396,85290,2498
1,Test,366620,33621,37850,2321



TRAINING BASKET SIZE SUMMARY


count   217,396.000
mean         10.030
std          12.420
min           1.000
25%           2.000
50%           5.000
75%          13.000
95%          36.000
99%          59.000
max         167.000
Name: unique_products, dtype: float64

In [42]:

# Load product information
product_query = text("""
    SELECT
        PRODUCT_ID,
        DEPARTMENT,
        COMMODITY_DESC,
        SUB_COMMODITY_DESC,
        BRAND,
        CURR_SIZE_OF_PRODUCT
    FROM raw_product
""")

with engine.connect() as connection:
    product_catalog = pd.read_sql_query(
        product_query,
        con=connection
    )

# Create readable product labels
product_catalog["product_label"] = (
    product_catalog["SUB_COMMODITY_DESC"]
    .fillna("Unknown Product")
    .astype(str)
    .str.strip()
    + " | "
    + product_catalog["CURR_SIZE_OF_PRODUCT"]
    .fillna("")
    .astype(str)
    .str.strip()
    + " | ID: "
    + product_catalog["PRODUCT_ID"].astype(str)
)

# Validate product catalog
assert product_catalog["PRODUCT_ID"].is_unique

print("Product Catalog Shape:", product_catalog.shape)

display(product_catalog.head(10))


Product Catalog Shape: (92353, 7)


,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,CURR_SIZE_OF_PRODUCT,product_label
0,25671,GROCERY,FRZN ICE,ICE - CRUSHED/CUBED,National,22 LB,ICE - CRUSHED/CUBED | 22 LB | ID: 25671
1,26081,MISC. TRANS.,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,National,,NO SUBCOMMODITY DESCRIPTION | | ID: 26081
2,26093,PASTRY,BREAD,BREAD:ITALIAN/FRENCH,Private,,BREAD:ITALIAN/FRENCH | | ID: 26093
3,26190,GROCERY,FRUIT - SHELF STABLE,APPLE SAUCE,Private,50 OZ,APPLE SAUCE | 50 OZ | ID: 26190
4,26355,GROCERY,COOKIES/CONES,SPECIALTY COOKIES,Private,14 OZ,SPECIALTY COOKIES | 14 OZ | ID: 26355
5,26426,GROCERY,SPICES & EXTRACTS,SPICES & SEASONINGS,Private,2.5 OZ,SPICES & SEASONINGS | 2.5 OZ | ID: 26426
6,26540,GROCERY,COOKIES/CONES,TRAY PACK/CHOC CHIP COOKIES,Private,16 OZ,TRAY PACK/CHOC CHIP COOKIES | 16 OZ | ID: 26540
7,26601,DRUG GM,VITAMINS,VITAMIN - MINERALS,Private,300CT(1),VITAMIN - MINERALS | 300CT(1) | ID: 26601
8,26636,PASTRY,BREAKFAST SWEETS,SW GDS: SW ROLLS/DAN,Private,,SW GDS: SW ROLLS/DAN | | ID: 26636
9,26691,GROCERY,PNT BTR/JELLY/JAMS,HONEY,Private,12 OZ,HONEY | 12 OZ | ID: 26691


* The "MISC. TRANS." category may not represent actual retail products. Let’s add this to our existing filters.
* Labels like "NO SUBCOMMODITY DESCRIPTION" are not user-friendly. Let’s use "COMMODITY_DESC" instead, whenever possible.

In [43]:
# Exclude non-retail departments
excluded_departments = [
    "KIOSK-GAS",
    "MISC SALES TRAN",
    "MISC. TRANS.",
    "COUP/STR & MFG",
]

valid_catalog = product_catalog.loc[
    ~product_catalog["DEPARTMENT"]
    .fillna("")
    .str.strip()
    .isin(excluded_departments)
].copy()

# Improve product descriptions
sub_desc = (
    valid_catalog["SUB_COMMODITY_DESC"]
    .fillna("")
    .str.strip()
)

commodity_desc = (
    valid_catalog["COMMODITY_DESC"]
    .fillna("")
    .str.strip()
)

valid_catalog["product_description"] = (
    sub_desc
    .replace({
        "": np.nan,
        "NO SUBCOMMODITY DESCRIPTION": np.nan,
    })
    .fillna(
        commodity_desc.replace({
            "": np.nan,
            "NO COMMODITY DESCRIPTION": np.nan,
        })
    )
    .fillna("Unknown Product")
)

# Create improved product labels
product_size = (
    valid_catalog["CURR_SIZE_OF_PRODUCT"]
    .fillna("")
    .str.strip()
)

valid_catalog["product_label"] = (
    valid_catalog["product_description"]
    + np.where(product_size.ne(""), " | " + product_size, "")
    + " | ID: "
    + valid_catalog["PRODUCT_ID"].astype(str)
)

# Keep only valid retail products in transaction datasets
valid_product_ids = set(valid_catalog["PRODUCT_ID"])

train_items = train_items.loc[
    train_items["PRODUCT_ID"].isin(valid_product_ids)
].copy()

test_items = test_items.loc[
    test_items["PRODUCT_ID"].isin(valid_product_ids)
].copy()

print("Valid Products:", len(valid_catalog))
print("Training Transactions:", len(train_items))
print("Test Transactions:", len(test_items))

display(valid_catalog.head())


Valid Products: 91720
Training Transactions: 2178447
Test Transactions: 366221


,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,CURR_SIZE_OF_PRODUCT,product_label,product_description
0,25671,GROCERY,FRZN ICE,ICE - CRUSHED/CUBED,National,22 LB,ICE - CRUSHED/CUBED | 22 LB | ID: 25671,ICE - CRUSHED/CUBED
2,26093,PASTRY,BREAD,BREAD:ITALIAN/FRENCH,Private,,BREAD:ITALIAN/FRENCH | ID: 26093,BREAD:ITALIAN/FRENCH
3,26190,GROCERY,FRUIT - SHELF STABLE,APPLE SAUCE,Private,50 OZ,APPLE SAUCE | 50 OZ | ID: 26190,APPLE SAUCE
4,26355,GROCERY,COOKIES/CONES,SPECIALTY COOKIES,Private,14 OZ,SPECIALTY COOKIES | 14 OZ | ID: 26355,SPECIALTY COOKIES
5,26426,GROCERY,SPICES & EXTRACTS,SPICES & SEASONINGS,Private,2.5 OZ,SPICES & SEASONINGS | 2.5 OZ | ID: 26426,SPICES & SEASONINGS



### 6.1.2 Product Frequency and Support Analysis

We will analyze product purchase frequency using training data.

Our goals are to:

- Identify frequently purchased products.
- Calculate product support values.
- Examine the distribution of product frequency.
- Select suitable minimum support thresholds.

Only training data will be used to select frequent products.


In [44]:
# Total number of training baskets
total_baskets = train_items["BASKET_ID"].nunique()

# Calculate product frequency
product_frequency = (
    train_items
    .groupby("PRODUCT_ID", as_index=False)
    .agg(
        basket_count=("BASKET_ID", "nunique")
    )
)

# Calculate support
product_frequency["support"] = (
    product_frequency["basket_count"] / total_baskets
)

# Add readable product information
product_frequency = product_frequency.merge(
    valid_catalog[
        ["PRODUCT_ID", "product_label", "DEPARTMENT"]
    ],
    on="PRODUCT_ID",
    how="left",
    validate="one_to_one"
)

# Sort products by frequency
product_frequency = (
    product_frequency
    .sort_values("basket_count", ascending=False)
    .reset_index(drop=True)
)

print("Total Training Baskets:", total_baskets)
print("Unique Purchased Products:", len(product_frequency))

display(product_frequency.head(15))

# Examine product frequency distribution
display(
    product_frequency["basket_count"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .round(2)
)

Total Training Baskets: 217341
Unique Purchased Products: 84879


,PRODUCT_ID,basket_count,support,product_label,DEPARTMENT
0,1082185,25541,0.118,BANANAS | 40 LB | ID: 1082185,PRODUCE
1,1029743,12553,0.058,FLUID MILK WHITE ONLY | 1 GA | ID: 1029743,GROCERY
2,995242,10503,0.048,FLUID MILK WHITE ONLY | ID: 995242,GROCERY
3,1106523,8543,0.039,FLUID MILK WHITE ONLY | 1 GA | ID: 1106523,GROCERY
4,981760,7905,0.036,EGGS - X-LARGE | 1 DZ | ID: 981760,GROCERY
5,1133018,6158,0.028,FLUID MILK WHITE ONLY | ID: 1133018,GROCERY
6,883404,5840,0.027,MAINSTREAM WHITE BREAD | 20 OZ | ID: 883404,GROCERY
7,951590,5283,0.024,MAINSTREAM WHITE BREAD | 20 OZ | ID: 951590,GROCERY
8,1127831,5151,0.024,STRAWBERRIES | 16 OZ | ID: 1127831,PRODUCE
9,826249,4949,0.023,HAMBURGER BUNS | 12 OZ | ID: 826249,GROCERY


count   84,879.000
mean        25.670
std        160.620
min          1.000
50%          3.000
75%         13.000
90%         54.000
95%        105.000
99%        328.000
max     25,541.000
Name: basket_count, dtype: float64

Support indicates the proportion of total baskets in which a product appears.


### 6.1.3 Minimum Support Threshold Analysis

We will compare different minimum support thresholds.

Our goals are to:

- Measure the number of eligible products.
- Measure basket coverage after filtering.
- Identify baskets with at least two eligible products.
- Select a practical threshold for association rule mining.

We will use training data only.


In [45]:
train_items.head()

,household_key,BASKET_ID,basket_day,PRODUCT_ID
0,1,27601281299,51,825123
1,1,27601281299,51,831447
2,1,27601281299,51,840361
3,1,27601281299,51,845307
4,1,27601281299,51,852014


In [46]:
# Define minimum support thresholds
support_thresholds = [0.01, 0.005, 0.002, 0.001]

threshold_results = []

for min_support in support_thresholds:

    # Select products above the support threshold
    frequent_products = product_frequency.loc[
        product_frequency["support"] >= min_support,
        "PRODUCT_ID"
    ]

    # Keep eligible basket-product relationships
    filtered_items = train_items.loc[
        train_items["PRODUCT_ID"].isin(frequent_products),
        ["BASKET_ID", "PRODUCT_ID"]
    ]

    # Count eligible products in each basket
    basket_product_counts = (
        filtered_items.groupby("BASKET_ID")["PRODUCT_ID"]
        .nunique()
    )

    # Count baskets with at least two eligible products
    baskets_with_pairs = (
        basket_product_counts >= 2
    ).sum()

    # Store results
    threshold_results.append({
        "min_support": min_support,
        "min_basket_count": int(
            np.ceil(min_support * total_baskets)
        ),
        "eligible_products": len(frequent_products),
        "baskets_with_pairs": baskets_with_pairs,
        "basket_coverage_pct": (
            baskets_with_pairs / total_baskets * 100
        ),
    })

# Create comparison table
support_evaluation = pd.DataFrame(threshold_results)

display(support_evaluation.round(3))

,min_support,min_basket_count,eligible_products,baskets_with_pairs,basket_coverage_pct
0,0.010,2174,53,57089,26.267
1,0.005,1087,140,77807,35.800
2,0.002,435,575,107951,49.669
3,0.001,218,1639,129159,59.427



### 6.1.4 FP-Growth — Frequent Itemset Mining

We selected a minimum support threshold of 0.002.

We will create a sparse basket-product matrix to reduce memory usage.

We will use FP-Growth to identify frequent product combinations.

For our first experiment, we will limit itemsets to a maximum of two products.

Support will be calculated using all training baskets.


In [47]:
# 1. Define minimum support
MIN_SUPPORT = 0.002

# 2. Select frequent products
frequent_product_ids = pd.Index(
    product_frequency.loc[
        product_frequency["support"] >= MIN_SUPPORT,
        "PRODUCT_ID"
    ].unique()
)

# 3. Keep frequent basket-product relationships
filtered_items = train_items.loc[
    train_items["PRODUCT_ID"].isin(frequent_product_ids),
    ["BASKET_ID", "PRODUCT_ID"]
].drop_duplicates()

# 4. Keep all training baskets
basket_ids = pd.Index(
    train_items["BASKET_ID"].unique()
)

# 5. Convert IDs into matrix positions
row_indices = basket_ids.get_indexer(
    filtered_items["BASKET_ID"]
)

col_indices = frequent_product_ids.get_indexer(
    filtered_items["PRODUCT_ID"]
)

# 6. Create sparse basket-product matrix
## A sparse matrix representing purchased products as True and others as False.
basket_matrix = csr_matrix(
    (
        np.ones(len(filtered_items), dtype=bool),
        (row_indices, col_indices)
    ),
    shape=(
        len(basket_ids),
        len(frequent_product_ids)
    )
)

# 7. Convert to a sparse Pandas DataFrame
basket_encoded = pd.DataFrame.sparse.from_spmatrix(
    basket_matrix,
    index=basket_ids,
    columns=frequent_product_ids
)

# 8. Validate matrix
assert basket_encoded.shape == (
    total_baskets,
    len(frequent_product_ids)
)

assert basket_matrix.nnz == len(filtered_items)

print("Basket Matrix Shape:", basket_encoded.shape)
print("Non-Zero Values:", basket_matrix.nnz)

print(
    "Matrix Density:",
    round(
        # Number of non-zero elements.
        basket_matrix.nnz / (
            basket_matrix.shape[0] * basket_matrix.shape[1]
        ) * 100,
        4
    ),
    "%"
)

Basket Matrix Shape: (217341, 575)
Non-Zero Values: 630747
Matrix Density: 0.5047 %


C:\Users\okand\AppData\Local\Temp\ipykernel_23224\147165948.py:46: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket_encoded = pd.DataFrame.sparse.from_spmatrix(


In [48]:
basket_encoded.head()

,1082185,1029743,995242,1106523,981760,1133018,883404,951590,1127831,826249,904360,1098066,840361,995785,908531,860776,961554,923746,5569230,1005186,1126899,849843,1070820,916122,854852,1044078,862349,1058997,844179,866211,1004906,1081177,878996,859075,1053690,1065593,986912,1085604,994928,834484,1096036,1068719,866227,844165,962229,962568,1024306,845208,903325,1033142,833025,847982,8090521,5568378,933835,909894,979707,899624,1092026,8090537,999971,5569471,1071939,893018,1038217,901062,965766,1080414,6034857,9526410,938700,1007195,1026118,1070015,879755,987562,1037863,983584,5592931,1006184,910032,1039156,1013321,914190,1056509,913785,1022254,1075368,953476,1101010,995965,847270,857849,1022003,1023720,1110572,1043590,909714,1062966,1003188,9527290,1120741,929668,824005,1028816,976998,908846,1124729,1041259,5585510,1055646,871756,1077555,986947,1105488,1101173,820165,951412,1051516,1136257,896369,940947,12810393,5569327,873627,927191,832678,907014,865178,1033615,913210,1137688,834117,965267,976199,1122358,868764,870547,1040807,916381,1119051,5569845,874972,989101,1042438,1110843,860975,890610,893501,1050229,861272,883932,999858,885290,1055863,990797,1114597,930917,908318,857503,1095275,1062002,971922,1134522,1132771,856772,985999,1043751,1003600,1048462,1018740,1023958,1102959,1027569,822407,833715,882441,1029624,970030,1117514,835098,1132770,1015247,880150,1137775,933637,9527494,1037840,894236,883003,1051069,1116821,897125,9337581,925514,861445,911812,1057260,930118,862139,854405,952317,872137,1058779,1053763,823704,900802,1050851,918335,870515,916993,825541,1051211,1043301,1070702,993638,999104,1041796,865456,1102949,866140,1002558,863447,839419,1107553,947798,12263788,12301109,1030819,987724,985119,1113588,897954,925626,892008,1106301,8090532,1087102,995876,819978,5569374,1118393,5568729,1138443,907631,1127179,1022872,918733,1037894,944534,999270,821083,1045220,1098248,9526411,1000753,1094955,1012587,1014948,5590287,1076875,1091365,1138189,1042907,968215,940766,1082212,973861,859010,12301100,1112238,879528,989075,838186,1118533,866871,921744,1069621,885863,1052912,867188,1096317,864774,1020581,863802,951526,1127624,898068,847789,9677202,970202,1055504,987480,902172,1075470,1046262,952163,874736,9527160,1056651,5584027,896085,1034686,852856,928049,1074405,9526563,8090509,1022053,951190,957951,949616,937626,974108,1077490,918368,883963,970119,931671,845307,972931,1074754,985893,944836,1027168,828867,1009449,8160319,1111072,1001148,1131438,913202,963971,1020682,1108168,1110793,943233,1133072,999779,965842,1054262,988693,7024990,8065410,957736,898980,824555,1051323,12301405,5568228,890695,849202,935284,8205218,845078,941741,839147,911878,998444,905539,7167218,910291,952981,948670,936594,1081189,967395,919534,944139,852486,935317,1100972,1033220,1096343,944137,1038663,831628,896613,1007191,1071342,5585635,922907,823758,9420312,5588666,980727,1012873,6602697,1022097,5584007,926905,920308,1003421,1130858,1046545,999714,870780,865528,892920,1011300,1074524,860299,873203,1073150,885697,858302,1084331,850841,1005609,901666,8090440,845193,1061220,976955,939636,969866,891954,1110142,882830,959985,986394,1115576,1077975,8090536,9835509,1036501,834103,6944571,1110244,12302069,892531,1111839,982493,982790,900491,981521,933067,1088835,844818,1105623,942525,839849,1022119,823990,1074333,906262,880427,1128016,898212,827683,938566,998119,934639,833723,1123792,944486,1050131,6442594,1068292,921504,937791,10121965,827180,1022981,1013928,1042544,922127,845705,819927,861279,819765,1104343,1037332,850102,856252,8019716,959409,9526886,8177622,857006,944317,9245106,5585735,934676,9553031,5568489,977499,896862,848029,5585727,7139529,956609,7025106,989824,945662,988791,993466,7440663,843756,5585639,9527487,1000237,873654,865705,985427,871611,823721,1044895,1135768,960732,1130777,909268,890536,1056746,1075214,837270,828106,1039589,948650,1105917,1118787,830750,1013770,895540,865174,874149,911215,891141,944466,1108131,1115175,889731,937292,1002771,113734

In [49]:
# Convert product IDs to string column names
basket_encoded.columns = basket_encoded.columns.astype(str)

# Verify column types
print("Column Type:", type(basket_encoded.columns[0]))
print("Matrix Shape:", basket_encoded.shape)

# Find frequent product combinations
frequent_itemsets = fpgrowth(
    basket_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=2
)

# Calculate itemset size
frequent_itemsets["itemset_size"] = (
    frequent_itemsets["itemsets"].apply(len)
)

# Sort by support
frequent_itemsets = (
    frequent_itemsets
    .sort_values("support", ascending=False)
    .reset_index(drop=True)
)

# Show itemset distribution
print("FREQUENT ITEMSET SUMMARY")

display(
    frequent_itemsets["itemset_size"]
    .value_counts()
    .sort_index()
)

# Show frequent product pairs
frequent_pairs = frequent_itemsets.loc[
    frequent_itemsets["itemset_size"] == 2
].copy()

print("\nFrequent Product Pairs:", len(frequent_pairs))

display(frequent_pairs.head(10))

Column Type: <class 'str'>
Matrix Shape: (217341, 575)
FREQUENT ITEMSET SUMMARY


itemset_size
1    575
2    136
Name: count, dtype: int64


Frequent Product Pairs: 136


,support,itemsets,itemset_size
33,0.014,"(1029743, 1082185)",2
46,0.012,"(995242, 1082185)",2
50,0.011,"(1082185, 1127831)",2
56,0.010,"(1082185, 981760)",2
67,0.008,"(1106523, 1082185)",2
83,0.007,"(1082185, 961554)",2
92,0.007,"(860776, 1082185)",2
93,0.007,"(1126899, 1082185)",2
94,0.007,"(1082185, 951590)",2
100,0.007,"(1098066, 826249)",2



### 6.1.5 Association Rule Generation

We will generate association rules from frequent product pairs.

We will evaluate the rules using Support, Confidence, and Lift.

Initial thresholds:
- Minimum Support: 0.002
- Minimum Confidence: 0.10
- Minimum Lift: 1.20

We will also add product descriptions to make the results easier to understand.

Notes:

- Support: The rate at which two products appear together.
- Confidence: The rate at which B is also purchased when A is purchased.
- Lift: How much more powerful purchasing together is compared to purchasing independently.

In [50]:
MIN_CONFIDENCE = 0.10
MIN_LIFT = 1.20

# Generate association rules
rules = association_rules(
    frequent_itemsets[["support", "itemsets"]],
    num_itemsets=len(basket_encoded),
    metric="confidence",
    min_threshold=MIN_CONFIDENCE,
)

# Filter rules
rules = rules.loc[
    (rules["lift"] >= MIN_LIFT)
    & (rules["support"] >= MIN_SUPPORT)
].copy()

# Create product label mapping
product_labels = valid_catalog.set_index(
    "PRODUCT_ID"
)["product_label"].to_dict()

# Convert product IDs to readable labels
# Convert product IDs to readable labels
def get_itemset_labels(itemset):
    return " + ".join(
        product_labels.get(
            int(product_id),
            f"Unknown Product: {product_id}"
        )
        for product_id in sorted(itemset, key=int)
    )

rules["antecedent_name"] = (
    rules["antecedents"].apply(get_itemset_labels)
)

rules["consequent_name"] = (
    rules["consequents"].apply(get_itemset_labels)
)

# Select important columns
association_results = rules[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
    ]
].sort_values(
    ["lift", "confidence"],
    ascending=False,
)

print("Total Association Rules:", len(association_results))

display(association_results.head(15).round(4))

Total Association Rules: 146


,antecedent_name,consequent_name,support,confidence,lift
106,BEANS GREEN: FS/WHL/CUT | 14.5 OZ | ID: 1013321,CORN | 15.25 OZ | ID: 938700,0.002,0.327,40.277
107,CORN | 15.25 OZ | ID: 938700,BEANS GREEN: FS/WHL/CUT | 14.5 OZ | ID: 1013321,0.002,0.287,40.277
63,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 8...,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 8...,0.003,0.282,31.686
64,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 8...,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 8...,0.003,0.323,31.686
125,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 5...,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 5...,0.002,0.437,23.385
124,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 5...,SOFT DRINKS 12/18&15PK CAN CAR | 12 OZ | ID: 5...,0.002,0.115,23.385
94,SFT DRNK 2 LITER BTL CARB INCL | 2 LTR | ID: 1...,SFT DRNK 2 LITER BTL CARB INCL | 2 LTR | ID: 8...,0.003,0.271,23.341
93,SFT DRNK 2 LITER BTL CARB INCL | 2 LTR | ID: 8...,SFT DRNK 2 LITER BTL CARB INCL | 2 LTR | ID: 1...,0.003,0.213,23.341
11,HAMBURGER BUNS | 12 OZ | ID: 826249,HOT DOG BUNS | 11 OZ | ID: 1098066,0.007,0.293,13.855
10,HOT DOG BUNS | 11 OZ | ID: 1098066,HAMBURGER BUNS | 12 OZ | ID: 826249,0.007,0.316,13.855



### 6.1.6 Association-Based Product Recommendation

We will create a product recommendation function using association rules.

The function will:
- Accept a product ID.
- Find related products.
- Rank recommendations using Confidence and Lift.
- Return readable product names.

We will use this function in our future FastAPI and Streamlit applications.


In [51]:
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_name,consequent_name
0,(1029743),(1082185),0.058,0.118,0.014,0.236,2.005,1.000,0.007,1.155,0.532,0.084,0.134,0.176,FLUID MILK WHITE ONLY | 1 GA | ID: 1029743,BANANAS | 40 LB | ID: 1082185
1,(1082185),(1029743),0.118,0.058,0.014,0.116,2.005,1.000,0.007,1.066,0.568,0.084,0.062,0.176,BANANAS | 40 LB | ID: 1082185,FLUID MILK WHITE ONLY | 1 GA | ID: 1029743
2,(995242),(1082185),0.048,0.118,0.012,0.238,2.029,1.000,0.006,1.159,0.533,0.075,0.137,0.168,FLUID MILK WHITE ONLY | ID: 995242,BANANAS | 40 LB | ID: 1082185
3,(1127831),(1082185),0.024,0.118,0.011,0.446,3.793,1.000,0.008,1.592,0.754,0.081,0.372,0.268,STRAWBERRIES | 16 OZ | ID: 1127831,BANANAS | 40 LB | ID: 1082185
4,(981760),(1082185),0.036,0.118,0.010,0.277,2.360,1.000,0.006,1.221,0.598,0.070,0.181,0.182,EGGS - X-LARGE | 1 DZ | ID: 981760,BANANAS | 40 LB | ID: 1082185


In [52]:
# Use rules with one antecedent and one consequent
pair_rules = rules.loc[
    rules["antecedents"].apply(len).eq(1)
    & rules["consequents"].apply(len).eq(1)
].copy()

# Extract product IDs
pair_rules["antecedent_id"] = (
    pair_rules["antecedents"]
    .apply(lambda x: int(next(iter(x))))
)

pair_rules["consequent_id"] = (
    pair_rules["consequents"]
    .apply(lambda x: int(next(iter(x))))
)

# Create recommendation dataset
recommendation_rules = pair_rules[
    [
        "antecedent_id",
        "consequent_id",
        "support",
        "confidence",
        "lift",
    ]
].copy()

# Add recommended product descriptions
recommendation_rules = recommendation_rules.merge(
    valid_catalog[
        ["PRODUCT_ID", "product_label"]
    ].rename(
        columns={
            "PRODUCT_ID": "consequent_id",
            "product_label": "recommended_product",
        }
    ),
    on="consequent_id",
    how="left",
    validate="many_to_one",
)

assert recommendation_rules["recommended_product"].notna().all()

print("Available Recommendation Rules:", len(recommendation_rules))

display(recommendation_rules.head())


Available Recommendation Rules: 146


,antecedent_id,consequent_id,support,confidence,lift,recommended_product
0,1029743,1082185,0.014,0.236,2.005,BANANAS | 40 LB | ID: 1082185
1,1082185,1029743,0.014,0.116,2.005,FLUID MILK WHITE ONLY | 1 GA | ID: 1029743
2,995242,1082185,0.012,0.238,2.029,BANANAS | 40 LB | ID: 1082185
3,1127831,1082185,0.011,0.446,3.793,BANANAS | 40 LB | ID: 1082185
4,981760,1082185,0.010,0.277,2.360,BANANAS | 40 LB | ID: 1082185


In [53]:
def recommend_products(
    product_id: int,
    top_n: int = 5,
) -> pd.DataFrame:
    """Recommend related products using association rules."""

    # Find rules starting with the selected product
    recommendations = recommendation_rules.loc[
        recommendation_rules["antecedent_id"] == product_id
    ].copy()

    # Rank products by Confidence and Lift
    recommendations = recommendations.sort_values(
        ["confidence", "lift", "support"],
        ascending=False,
    )

    # Remove duplicate recommendations
    recommendations = recommendations.drop_duplicates(
        subset="consequent_id"
    )

    # Return top recommendations
    return (
        recommendations[
            [
                "consequent_id",
                "recommended_product",
                "support",
                "confidence",
                "lift",
            ]
        ]
        .head(top_n)
        .reset_index(drop=True)
    )

In [54]:
valid_catalog.loc[valid_catalog["PRODUCT_ID"] == 826249, "product_label"]

6677    HAMBURGER BUNS | 12 OZ | ID: 826249
Name: product_label, dtype: object

In [55]:
# Example: Hamburger Buns
PRODUCT_ID = 826249

selected_product= valid_catalog.loc[
    valid_catalog["PRODUCT_ID"] == PRODUCT_ID,
    "product_label"
].iloc[0]

print("Selected Product:", selected_product)

# Generate recommendations
product_recommendations = recommend_products(
    product_id=PRODUCT_ID,
    top_n=5
)

display(product_recommendations.round(4))

Selected Product: HAMBURGER BUNS | 12 OZ | ID: 826249


,consequent_id,recommended_product,support,confidence,lift
0,1098066,HOT DOG BUNS | 11 OZ | ID: 1098066,0.007,0.293,13.855
1,1082185,BANANAS | 40 LB | ID: 1082185,0.005,0.226,1.919
2,1029743,FLUID MILK WHITE ONLY | 1 GA | ID: 1029743,0.003,0.134,2.323
3,883404,MAINSTREAM WHITE BREAD | 20 OZ | ID: 883404,0.003,0.124,4.625


In [56]:
# Save association rules
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

recommendation_rules.to_csv(
    output_dir / "association_rules.csv",
    index=False
)

print("Association Rules Saved:", len(recommendation_rules))

Association Rules Saved: 146



## 6.2 Item-Based Collaborative Filtering

Item-Based Collaborative Filtering recommends products based on similarities between items.

We will use customer purchase histories to calculate product similarities.

Our approach:

- Create a customer-product interaction matrix.
- Use binary purchase indicators.
- Filter products with very few customers.
- Calculate Cosine Similarity.
- Find similar products using Nearest Neighbors.

We will use sparse matrices to reduce memory usage.


In [57]:
train_items.head()

,household_key,BASKET_ID,basket_day,PRODUCT_ID
0,1,27601281299,51,825123
1,1,27601281299,51,831447
2,1,27601281299,51,840361
3,1,27601281299,51,845307
4,1,27601281299,51,852014


In [58]:
# 1. Create unique customer-product interactions
customer_product = train_items[
    ["household_key", "PRODUCT_ID"]
].drop_duplicates()

customer_product.head(10)

,household_key,PRODUCT_ID
0,1,825123
1,1,831447
2,1,840361
3,1,845307
4,1,852014
5,1,854982
6,1,856942
7,1,912676
8,1,940947
9,1,945805


In [59]:
# 2. Count unique customers per product
product_customer_counts = (
    customer_product
    .groupby("PRODUCT_ID")["household_key"]
    .nunique()
)

product_customer_counts.head(10)

PRODUCT_ID
25671    3
26093    1
26190    1
26355    1
26426    1
26540    1
26601    1
26636    1
26691    2
26738    1
Name: household_key, dtype: int64

In [60]:
# 3. Define minimum customer threshold
MIN_CUSTOMERS = 25

eligible_products = product_customer_counts.loc[
    product_customer_counts >= MIN_CUSTOMERS
].index

# 4. Filter interactions
cf_interactions = customer_product.loc[
    customer_product["PRODUCT_ID"].isin(eligible_products)
].copy()

# 5. Create customer and product indexes
customer_ids = pd.Index(
    sorted(cf_interactions["household_key"].unique())
)

cf_product_ids = pd.Index(
    sorted(cf_interactions["PRODUCT_ID"].unique())
)

# 6. Map IDs to matrix positions
row_indices = customer_ids.get_indexer(
    cf_interactions["household_key"]
)

col_indices = cf_product_ids.get_indexer(
    cf_interactions["PRODUCT_ID"]
)

# 7. Create sparse customer-product matrix
customer_item_matrix = csr_matrix(
    (
        np.ones(len(cf_interactions), dtype=np.float32),
        (row_indices, col_indices)
    ),
    shape=(len(customer_ids), len(cf_product_ids)),
    dtype=np.float32
)

# 8. Validate matrix
assert customer_item_matrix.nnz == len(cf_interactions)

print("Customer-Product Matrix:", customer_item_matrix.shape)
print("Unique Customers:", len(customer_ids))
print("Eligible Products:", len(cf_product_ids))
print("Non-Zero Interactions:", customer_item_matrix.nnz)

print(
    "Matrix Density:",
    round(
        customer_item_matrix.nnz
        / (customer_item_matrix.shape[0] * customer_item_matrix.shape[1])
        * 100,
        4
    ),
    "%"
)

Customer-Product Matrix: (2481, 11603)
Unique Customers: 2481
Eligible Products: 11603
Non-Zero Interactions: 917897
Matrix Density: 3.1886 %


In [61]:
item_user_matrix = customer_item_matrix.T.tocsr()


### 6.2.1 Item Similarity with Nearest Neighbors

We will use Cosine Similarity to find products with similar customer purchase patterns.

We will train a Nearest Neighbors model using the item-user matrix.

The model will identify similar products without creating a full product similarity matrix.


In [62]:
# Initialize Item-Based Nearest Neighbors
item_knn =NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

# Fit the model
item_knn.fit(item_user_matrix)

print("Item-Based Nearest Neighbors model fitted.")
print("Number of Products:", item_user_matrix.shape[0])
print("Number of Customers:", item_user_matrix.shape[1])

Item-Based Nearest Neighbors model fitted.
Number of Products: 11603
Number of Customers: 2481


In [63]:
def recommend_similar_items(
    product_id: int,
    top_n: int = 5,
) -> pd.DataFrame:
    """Recommend similar products using Item-Based KNN."""

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    # Find the selected product's matrix position
    product_idx = cf_product_ids.get_indexer([product_id])[0]

    if product_idx == -1:
        raise ValueError("Product is not available in the model.")

    # Find nearest products
    distances, indices = item_knn.kneighbors(
        item_user_matrix[product_idx],
        n_neighbors=min(top_n + 1, len(cf_product_ids)),
    )

    # Create recommendation results
    results = pd.DataFrame({
        "PRODUCT_ID": cf_product_ids.take(indices[0]),
        "cosine_similarity": 1 - distances[0],
    })

    # Remove the selected product itself
    results = results.loc[
        results["PRODUCT_ID"] != product_id
    ].head(top_n).copy()

    # Add product labels
    results = results.merge(
        valid_catalog[
            ["PRODUCT_ID", "product_label"]
        ],
        on="PRODUCT_ID",
        how="left",
        validate="one_to_one",
    )

    return results[
        ["PRODUCT_ID", "product_label", "cosine_similarity"]
    ].reset_index(drop=True)


In [64]:
# Example: Hamburger Buns
PRODUCT_ID = 826249

print(
    "Selected Product:",
    valid_catalog.loc[
        valid_catalog["PRODUCT_ID"] == PRODUCT_ID,
        "product_label"
    ].iloc[0]
)

similar_items = recommend_similar_items(
    product_id=PRODUCT_ID,
    top_n=5,
)

display(similar_items.round(4))

Selected Product: HAMBURGER BUNS | 12 OZ | ID: 826249


,PRODUCT_ID,product_label,cosine_similarity
0,1098066,HOT DOG BUNS | 11 OZ | ID: 1098066,0.764
1,1082185,BANANAS | 40 LB | ID: 1082185,0.690
2,981760,EGGS - X-LARGE | 1 DZ | ID: 981760,0.677
3,883404,MAINSTREAM WHITE BREAD | 20 OZ | ID: 883404,0.660
4,1029743,FLUID MILK WHITE ONLY | 1 GA | ID: 1029743,0.631



### 6.2.2 Save Item-Based Collaborative Filtering Model

We will save the trained Item-Based KNN model and its related data.

This will allow us to reuse the model without training it again.

The saved files will be used in our future FastAPI and Streamlit applications.


In [65]:
# Create the model directory
model_dir = PROJECT_ROOT / "models" / "recommendation"
model_dir.mkdir(parents=True, exist_ok=True)

# 1. Save the trained KNN model
joblib.dump(
    item_knn,
    model_dir / "item_knn.joblib"
)

# 2. Save the sparse item-user matrix
save_npz(
    model_dir / "item_user_matrix.npz",
    item_user_matrix
)

# 3. Save product IDs and model configuration
item_metadata = {
    "product_ids": cf_product_ids.tolist(),
    "min_customers": MIN_CUSTOMERS,
    "similarity_metric": "cosine",
    "training_end_day": TRAIN_END_DAY,
}

joblib.dump(
    item_metadata,
    model_dir / "item_based_metadata.joblib"
)

print("Item-Based model artifacts saved successfully.")


Item-Based model artifacts saved successfully.



## 6.3 User-Based Collaborative Filtering

User-Based Collaborative Filtering recommends products based on similar customers.

We will use the customer-product interaction matrix created earlier.

Our approach:

- Calculate customer similarity using Cosine Similarity.
- Find the nearest customers with KNN.
- Identify products purchased by similar customers.
- Exclude products already purchased by the target customer.
- Rank recommended products using similarity-weighted scores.

We will use only training data to generate recommendations.

In [66]:
# Initialize User-Based KNN model
user_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

# Fit the customer-product matrix
user_knn.fit(customer_item_matrix)

print("User-Based KNN model fitted.")
print("Customers:", customer_item_matrix.shape[0])
print("Products:", customer_item_matrix.shape[1])

User-Based KNN model fitted.
Customers: 2481
Products: 11603


In [67]:
def recommend_user_based(
    household_key: int,
    top_n: int = 5,
    n_neighbors: int = 30,
) -> pd.DataFrame:
    """Recommend products using similar customers."""

    if top_n < 1 or n_neighbors < 1:
        raise ValueError("top_n and n_neighbors must be positive.")

    # 1. Find customer index
    customer_idx = customer_ids.get_indexer(
        [household_key]
    )[0]

    if customer_idx == -1:
        raise ValueError("Customer is not available in the model.")

    # 2. Find similar customers
    distances, indices = user_knn.kneighbors(
        customer_item_matrix[customer_idx],
        n_neighbors=min(n_neighbors + 1, len(customer_ids)),
    )

    neighbor_indices = indices[0]
    similarities = 1 - distances[0]

    # 3. Exclude the customer and zero-similarity neighbors
    mask = (
        (neighbor_indices != customer_idx)
        & (similarities > 0)
    )

    neighbor_indices = neighbor_indices[mask]
    similarities = similarities[mask]

    columns = ["PRODUCT_ID", "product_label", "recommendation_score"]

    if len(neighbor_indices) == 0:
        return pd.DataFrame(columns=columns)

    # 4. Calculate similarity-weighted product scores
    neighbor_purchases = customer_item_matrix[
        neighbor_indices
    ]

    scores = np.asarray(
        neighbor_purchases.T @ similarities
    ).ravel()

    scores = scores / similarities.sum()

    # 5. Exclude products already purchased
    purchased_indices = customer_item_matrix[
        customer_idx
    ].indices

    scores[purchased_indices] = 0

    # 6. Select products with positive scores
    candidate_indices = np.flatnonzero(scores > 0)

    top_indices = candidate_indices[
        np.argsort(scores[candidate_indices])[::-1][:top_n]
    ]

    # 7. Create recommendations
    recommendations = pd.DataFrame({
        "PRODUCT_ID": cf_product_ids.take(top_indices),
        "recommendation_score": scores[top_indices],
    })

    # 8. Add product descriptions
    recommendations = recommendations.merge(
        valid_catalog[["PRODUCT_ID", "product_label"]],
        on="PRODUCT_ID",
        how="left",
        validate="one_to_one",
    )

    return recommendations[columns].reset_index(drop=True)

In [68]:

# Select an example customer
CUSTOMER_ID = int(customer_ids[0])

# Generate recommendations
user_recommendations = recommend_user_based(
    household_key=CUSTOMER_ID,
    top_n=10,
    n_neighbors=30,
)

print("Selected Customer:", CUSTOMER_ID)

print(
    "Previously Purchased Products:",
    customer_item_matrix[
        customer_ids.get_loc(CUSTOMER_ID)
    ].nnz
)

display(user_recommendations.round(4))


Selected Customer: 1
Previously Purchased Products: 400


,PRODUCT_ID,product_label,recommendation_score
0,1127831,STRAWBERRIES | 16 OZ | ID: 1127831,1.000
1,854852,TOMATOES HOTHOUSE ON THE VINE | 13 LB | ID: 85...,0.900
2,1024306,CANTALOUPE WHOLE | ID: 1024306,0.869
3,962568,CELERY | ID: 962568,0.838
4,981760,EGGS - X-LARGE | 1 DZ | ID: 981760,0.829
5,866211,GRAPES WHITE | 18 LB | ID: 866211,0.803
6,878996,GRAPES RED | 18 LB | ID: 878996,0.803
7,834484,ONIONS OTHER | 48 CT | ID: 834484,0.740
8,1023720,PEACHES YELLOW FLESH | ID: 1023720,0.732
9,1007195,CORN YELLOW | 48 CT | ID: 1007195,0.730



### 6.3.2 Save User-Based Collaborative Filtering Model

We will save the trained User-Based KNN model, customer-product matrix, and metadata.

These files will be used later in FastAPI and Streamlit.


In [69]:

# Model directory
model_dir = PROJECT_ROOT / "models" / "recommendation"
model_dir.mkdir(parents=True, exist_ok=True)

# Save trained model
joblib.dump(
    user_knn,
    model_dir / "user_knn.joblib"
)

# Save sparse customer-product matrix
save_npz(
    model_dir / "customer_item_matrix.npz",
    customer_item_matrix
)

# Save metadata
user_metadata = {
    "customer_ids": customer_ids.tolist(),
    "product_ids": cf_product_ids.tolist(),
    "min_customers": MIN_CUSTOMERS,
    "training_end_day": TRAIN_END_DAY,
    "similarity_metric": "cosine",
}

joblib.dump(
    user_metadata,
    model_dir / "user_based_metadata.joblib"
)

print("User-Based model artifacts saved successfully.")


User-Based model artifacts saved successfully.



## 6.4 Content-Based Filtering

Content-Based Filtering recommends products based on their features.

We will use product categories and descriptions to identify similar products.

Our approach:

1. Prepare product text features.
2. Apply TF-IDF Vectorization.
3. Calculate Cosine Similarity.
4. Generate similar product recommendations.

We will initially use the same product catalog as Item-Based Collaborative Filtering.


In [70]:
valid_catalog.head()

,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,CURR_SIZE_OF_PRODUCT,product_label,product_description
0,25671,GROCERY,FRZN ICE,ICE - CRUSHED/CUBED,National,22 LB,ICE - CRUSHED/CUBED | 22 LB | ID: 25671,ICE - CRUSHED/CUBED
2,26093,PASTRY,BREAD,BREAD:ITALIAN/FRENCH,Private,,BREAD:ITALIAN/FRENCH | ID: 26093,BREAD:ITALIAN/FRENCH
3,26190,GROCERY,FRUIT - SHELF STABLE,APPLE SAUCE,Private,50 OZ,APPLE SAUCE | 50 OZ | ID: 26190,APPLE SAUCE
4,26355,GROCERY,COOKIES/CONES,SPECIALTY COOKIES,Private,14 OZ,SPECIALTY COOKIES | 14 OZ | ID: 26355,SPECIALTY COOKIES
5,26426,GROCERY,SPICES & EXTRACTS,SPICES & SEASONINGS,Private,2.5 OZ,SPICES & SEASONINGS | 2.5 OZ | ID: 26426,SPICES & SEASONINGS


In [74]:
# 1. Select products used by our CF model
content_catalog = (
    valid_catalog
    .set_index("PRODUCT_ID")
    .loc[cf_product_ids]
    .rename_axis("PRODUCT_ID")
    .reset_index()
    .copy()
)

# 2. Select product features
content_features = [
    "DEPARTMENT",
    "COMMODITY_DESC",
    "SUB_COMMODITY_DESC",
]

# 3. Clean missing and unknown descriptions
for col in content_features:
    content_catalog[col] = (
        content_catalog[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .replace({
            "NO COMMODITY DESCRIPTION": "",
            "NO SUBCOMMODITY DESCRIPTION": "",
        })
    )

# 4. Combine product features into text
content_catalog["content_text"] = (
    content_catalog[content_features]
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 5. Validate results
assert content_catalog["PRODUCT_ID"].is_unique
assert len(content_catalog) == len(cf_product_ids)
assert content_catalog["content_text"].ne("").all()

print("Content Catalog Shape:", content_catalog.shape)

display(
    content_catalog[
        ["PRODUCT_ID", "product_label", "content_text"]
    ].head(10)
)

Content Catalog Shape: (11603, 9)


,PRODUCT_ID,product_label,content_text
0,201704,BANANAS | 40 LB | ID: 201704,PRODUCE TROPICAL FRUIT BANANAS
1,818980,PREPARED BEANS - BAKED W/PORK | 15.75 OZ | ID:...,GROCERY BEANS - CANNED GLASS & MW PREPARED BEA...
2,818981,ALL FAMILY CEREAL | 10.4 OZ | ID: 818981,GROCERY COLD CEREAL ALL FAMILY CEREAL
3,818996,TREATS | 6.2 OZ | ID: 818996,GROCERY CONVENIENT BRKFST/WHLSM SNACKS TREATS
4,819063,MAINSTREAM | 26 OZ | ID: 819063,GROCERY PASTA SAUCE MAINSTREAM
5,819112,TOMATO JUICE (OVER 50% JCE) | 46 OZ | ID: 819112,GROCERY CANNED JUICES TOMATO JUICE (OVER 50% JCE)
6,819255,ROLLS - PORK | 1 LB | ID: 819255,MEAT-PCKGD BREAKFAST SAUSAGE/SANDWICHES ROLLS ...
7,819304,MAINSTREAM | 26 OZ | ID: 819304,GROCERY PASTA SAUCE MAINSTREAM
8,819308,CHOICE BEEF | ID: 819308,MEAT BEEF CHOICE BEEF
9,819330,GRANOLA BARS | 10 OZ | ID: 819330,GROCERY CONVENIENT BRKFST/WHLSM SNACKS GRANOLA...


In [76]:
# Initialize TF-IDF
tfidf = TfidfVectorizer(
    # Single words and two-word phrases
    ngram_range=(1,2),
    # Terms found in at least 2 product descriptions
    min_df=2,
    # Maximum of 20,000 text characters
    max_features=20000,
    # It reduces the impact of word repetitions logarithmically.
    sublinear_tf=True,
    dtype=np.float32
)

# Fit and transform product descriptions
content_matrix = tfidf.fit_transform(
    content_catalog["content_text"]
)

print("TF-IDF Matrix Shape:", content_matrix.shape)
print("Vocabulary Size:", len(tfidf.vocabulary_))
print("Non-Zero Values:", content_matrix.nnz)

display(
    pd.DataFrame({
        "feature": tfidf.get_feature_names_out()
    }).head(20)
)

TF-IDF Matrix Shape: (11603, 3542)
Vocabulary Size: 3542
Non-Zero Values: 133062


,feature
0,10
1,10 juice
2,100
3,100 juice
4,100 pure
5,12
6,12 18
7,15lbs
8,15pk
9,15pk can



### 6.4.3 Content-Based Similarity Model

We will use Cosine Similarity to identify products with similar descriptions.

Each product is represented by a TF-IDF vector.

We will use the Nearest Neighbors algorithm to find similar products without creating a full similarity matrix.

Our model will use product information rather than customer purchase history.


In [77]:
# 1. Create product index
content_product_ids = pd.Index(
    content_catalog["PRODUCT_ID"]
)

# 2. Initialize Content-Based KNN model
content_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

# 3. Fit the model
content_knn.fit(content_matrix)

# 4. Validate matrix dimensions
assert content_matrix.shape[0] == len(content_product_ids)

print("Content-Based KNN model fitted.")
print("Products:", content_matrix.shape[0])
print("TF-IDF Features:", content_matrix.shape[1])

Content-Based KNN model fitted.
Products: 11603
TF-IDF Features: 3542



### 6.4.4 Content-Based Product Recommendations

We will create a recommendation function using product descriptions.

The function will:

- Accept a product ID.
- Find the nearest products using Cosine Similarity.
- Exclude the selected product.
- Return the most similar products.
- Include readable product labels.

We will use this function in our future application.


In [79]:
def recommend_content_based(
    product_id: int,
    top_n: int = 5,
) -> pd.DataFrame:
    """Recommend similar products using TF-IDF and Cosine Similarity."""

    if top_n < 1:
        raise ValueError("top_n must be positive.")

    # 1. Find the product's matrix position
    product_idx = content_product_ids.get_indexer(
        [product_id]
    )[0]

    if product_idx == -1:
        raise ValueError("Product is not available in the model.")

    # 2. Find nearest products
    n_neighbors = min(
        top_n + 20,
        len(content_product_ids),
    )

    distances, indices = content_knn.kneighbors(
        content_matrix[product_idx],
        n_neighbors=n_neighbors,
    )

    # 3. Calculate similarity scores
    recommendations = pd.DataFrame({
        "PRODUCT_ID": content_product_ids.take(indices[0]),
        "cosine_similarity": 1 - distances[0],
    })

    # 4. Remove the selected product and zero similarity
    recommendations = recommendations.loc[
        (recommendations["PRODUCT_ID"] != product_id)
        & (recommendations["cosine_similarity"] > 0)
    ]

    # 5. Select top recommendations
    recommendations = (
        recommendations
        .sort_values("cosine_similarity", ascending=False)
        .head(top_n)
        .copy()
    )

    # 6. Add product descriptions
    recommendations = recommendations.merge(
        content_catalog[
            ["PRODUCT_ID", "product_label", "COMMODITY_DESC"]
        ],
        on="PRODUCT_ID",
        how="left",
        validate="one_to_one",
    )

    return recommendations[
        [
            "PRODUCT_ID",
            "product_label",
            "COMMODITY_DESC",
            "cosine_similarity",
        ]
    ].reset_index(drop=True)



In [80]:
# Select an example product
PRODUCT_ID = 826249

# Display selected product
selected_product = content_catalog.loc[
    content_catalog["PRODUCT_ID"] == PRODUCT_ID,
    "product_label"
].iloc[0]

print("Selected Product:", selected_product)

# Generate Content-Based recommendations
content_recommendations = recommend_content_based(
    product_id=PRODUCT_ID,
    top_n=10,
)

display(content_recommendations.round(4))


Selected Product: HAMBURGER BUNS | 12 OZ | ID: 826249


,PRODUCT_ID,product_label,COMMODITY_DESC,cosine_similarity
0,1055412,HAMBURGER BUNS | 21 OZ | ID: 1055412,BAKED BREAD/BUNS/ROLLS,1.000
1,948509,HAMBURGER BUNS | 13 OZ | ID: 948509,BAKED BREAD/BUNS/ROLLS,1.000
2,1087916,HAMBURGER BUNS | 21 OZ | ID: 1087916,BAKED BREAD/BUNS/ROLLS,1.000
3,999714,HAMBURGER BUNS | ID: 999714,BAKED BREAD/BUNS/ROLLS,1.000
4,896444,HAMBURGER BUNS | 12 OZ | ID: 896444,BAKED BREAD/BUNS/ROLLS,1.000
5,1105616,HAMBURGER BUNS | 21 OZ | ID: 1105616,BAKED BREAD/BUNS/ROLLS,1.000
6,835694,HAMBURGER BUNS | 16 OZ | ID: 835694,BAKED BREAD/BUNS/ROLLS,1.000
7,850801,HAMBURGER BUNS | 21 OZ | ID: 850801,BAKED BREAD/BUNS/ROLLS,1.000
8,1096792,HAMBURGER BUNS | 12 OZ | ID: 1096792,BAKED BREAD/BUNS/ROLLS,1.000
9,873902,HAMBURGER BUNS | 15 OZ | ID: 873902,BAKED BREAD/BUNS/ROLLS,1.000



### 6.4.5 Improving Content-Based Features

Our first Content-Based model gave identical similarity scores to several product variants.

This happened because products had the same category descriptions.

We will improve the feature representation by adding:

- Brand information
- Product size information

We will use ColumnTransformer to combine TF-IDF and One-Hot Encoding.

Our goal is to make product similarities more informative.


In [82]:
content_catalog.head()

,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,SUB_COMMODITY_DESC,BRAND,CURR_SIZE_OF_PRODUCT,product_label,product_description,content_text
0,201704,PRODUCE,TROPICAL FRUIT,BANANAS,National,40 LB,BANANAS | 40 LB | ID: 201704,BANANAS,PRODUCE TROPICAL FRUIT BANANAS
1,818980,GROCERY,BEANS - CANNED GLASS & MW,PREPARED BEANS - BAKED W/PORK,National,15.75 OZ,PREPARED BEANS - BAKED W/PORK | 15.75 OZ | ID:...,PREPARED BEANS - BAKED W/PORK,GROCERY BEANS - CANNED GLASS & MW PREPARED BEA...
2,818981,GROCERY,COLD CEREAL,ALL FAMILY CEREAL,National,10.4 OZ,ALL FAMILY CEREAL | 10.4 OZ | ID: 818981,ALL FAMILY CEREAL,GROCERY COLD CEREAL ALL FAMILY CEREAL
3,818996,GROCERY,CONVENIENT BRKFST/WHLSM SNACKS,TREATS,National,6.2 OZ,TREATS | 6.2 OZ | ID: 818996,TREATS,GROCERY CONVENIENT BRKFST/WHLSM SNACKS TREATS
4,819063,GROCERY,PASTA SAUCE,MAINSTREAM,National,26 OZ,MAINSTREAM | 26 OZ | ID: 819063,MAINSTREAM,GROCERY PASTA SAUCE MAINSTREAM


In [84]:
# 1. Prepare categorical features
categorical_features = [
    "BRAND",
    "CURR_SIZE_OF_PRODUCT",
]

for col in categorical_features:
    content_catalog[col] = (
        content_catalog[col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
        .replace("", "UNKNOWN")
    )

# 2. Create the preprocessing pipeline
content_preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_features=20000,
                sublinear_tf=True,
                dtype=np.float32,
            ),
            "content_text",
        ),
        (
            "brand",
            OneHotEncoder(handle_unknown="ignore"),
            ["BRAND"],
        ),
        (
            "size",
            OneHotEncoder(handle_unknown="ignore"),
            ["CURR_SIZE_OF_PRODUCT"],
        )
    ],
    # These weights allow us to control the influence of feature groups on the Cosine Similarity calculation.
    transformer_weights={
        "text": 2.0,
        "brand": 0.5,
        "size": 1.0,
    },
    sparse_threshold=1.0,
)

# 3. Transform product features
# We are converting the ColumnTransformer output into CSR sparse matrix format.
content_matrix = content_preprocessor.fit_transform(
    content_catalog
).tocsr()

# 4. Update the similarity model
content_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

content_knn.fit(content_matrix)

print("Improved Content Matrix:", content_matrix.shape)
print("Non-Zero Values:", content_matrix.nnz)

# 5. Generate recommendations
content_recommendations_v2 = recommend_content_based(
    product_id=826249,
    top_n=10,
)

display(content_recommendations_v2.round(4))

Improved Content Matrix: (11603, 4712)
Non-Zero Values: 156268


,PRODUCT_ID,product_label,COMMODITY_DESC,cosine_similarity
0,883039,HAMBURGER BUNS | 12 OZ | ID: 883039,BAKED BREAD/BUNS/ROLLS,1.000
1,896444,HAMBURGER BUNS | 12 OZ | ID: 896444,BAKED BREAD/BUNS/ROLLS,1.000
2,1096792,HAMBURGER BUNS | 12 OZ | ID: 1096792,BAKED BREAD/BUNS/ROLLS,1.000
3,6904925,HAMBURGER BUNS | 12 OZ | ID: 6904925,BAKED BREAD/BUNS/ROLLS,1.000
4,1068315,HAMBURGER BUNS | 12 OZ | ID: 1068315,BAKED BREAD/BUNS/ROLLS,0.952
5,1087916,HAMBURGER BUNS | 21 OZ | ID: 1087916,BAKED BREAD/BUNS/ROLLS,0.809
6,12324956,HAMBURGER BUNS | 16CT/26 OZ | ID: 12324956,BAKED BREAD/BUNS/ROLLS,0.809
7,12324954,HAMBURGER BUNS | 16CT/24 OZ | ID: 12324954,BAKED BREAD/BUNS/ROLLS,0.809
8,15506577,HAMBURGER BUNS | 16 OZ | ID: 15506577,BAKED BREAD/BUNS/ROLLS,0.809
9,835694,HAMBURGER BUNS | 16 OZ | ID: 835694,BAKED BREAD/BUNS/ROLLS,0.809



### 6.4.6 Save Content-Based Model

We will save the trained Content-Based recommendation model.

The saved artifacts include:

- Feature preprocessing pipeline
- Nearest Neighbors model
- TF-IDF and categorical feature matrix
- Product identifiers and descriptions

We will reuse these files in our future application.


In [85]:
# Create model directory
model_dir = PROJECT_ROOT / "models" / "recommendation"
model_dir.mkdir(parents=True, exist_ok=True)

# 1. Save fitted preprocessing pipeline
joblib.dump(
    content_preprocessor,
    model_dir / "content_preprocessor.joblib"
)

# 2. Save fitted KNN model
joblib.dump(
    content_knn,
    model_dir / "content_knn.joblib"
)

# 3. Save sparse feature matrix
save_npz(
    model_dir / "content_matrix.npz",
    content_matrix
)

# 4. Save product metadata
content_metadata = {
    "product_ids": content_product_ids.tolist(),
    "similarity_metric": "cosine",
    "training_end_day": TRAIN_END_DAY,
    "feature_groups": [
        "content_text",
        "BRAND",
        "CURR_SIZE_OF_PRODUCT",
    ],
}

joblib.dump(
    content_metadata,
    model_dir / "content_based_metadata.joblib"
)

# 5. Save readable product information
content_catalog[
    [
        "PRODUCT_ID",
        "product_label",
        "COMMODITY_DESC",
        "BRAND",
        "CURR_SIZE_OF_PRODUCT",
    ]
].to_csv(
    model_dir / "content_product_catalog.csv",
    index=False,
)

print("Content-Based model artifacts saved.")

Content-Based model artifacts saved.



## 6.5 Matrix Factorization

### 6.5.1 Truncated SVD

We will use Truncated SVD to learn lower-dimensional representations of customers and products.

Our customer-product matrix contains binary purchase interactions.

Truncated SVD can work directly with sparse matrices.

We will start with 50 latent components.

The model will be trained using historical purchase data only.


In [88]:
# 1. Define latent dimensions
N_COMPONENTS = 50

# 2. Initialize Truncated SVD
svd_model = TruncatedSVD(
    n_components=N_COMPONENTS,
    n_iter=7,
    random_state=42,
)

# 3. Learn customer representations
user_factors = svd_model.fit_transform(
    customer_item_matrix
)

# 4. Get item representations
item_factors = svd_model.components_.T

# 5. Validate dimensions
assert user_factors.shape == (
    len(customer_ids),
    N_COMPONENTS,
)

assert item_factors.shape == (
    len(cf_product_ids),
    N_COMPONENTS,
)

print("Original Matrix:", customer_item_matrix.shape)
print("User Factors:", user_factors.shape)
print("Item Factors:", item_factors.shape)

print(
    "Explained Variance Ratio:",
    round(svd_model.explained_variance_ratio_.sum(), 4)
)

Original Matrix: (2481, 11603)
User Factors: (2481, 50)
Item Factors: (11603, 50)
Explained Variance Ratio: 0.1563



### 6.5.2 SVD-Based Product Recommendations

We will use the learned customer and product factors to calculate recommendation scores.

The function will:

- Find the selected customer.
- Calculate scores for all available products.
- Remove previously purchased products.
- Rank products by their predicted scores.
- Return the top recommendations.

SVD scores represent estimated interactions, not purchase probabilities.


In [89]:
def recommend_svd(
    household_key: int,
    top_n: int = 10,
) -> pd.DataFrame:
    """Recommend products using Truncated SVD factors."""

    if top_n < 1:
        raise ValueError("top_n must be positive.")

    # 1. Find customer index
    customer_idx = customer_ids.get_indexer(
        [household_key]
    )[0]

    if customer_idx == -1:
        raise ValueError("Customer is not available in the model.")

    # 2. Calculate product scores for the selected customer
    scores = np.asarray(
        user_factors[customer_idx] @ item_factors.T
    ).ravel()

    # 3. Find previously purchased products
    purchased_indices = customer_item_matrix[
        customer_idx
    ].indices

    # 4. Exclude previously purchased products
    scores[purchased_indices] = -np.inf

    # 5. Keep valid positive scores
    candidate_indices = np.flatnonzero(
        np.isfinite(scores) & (scores > 0)
    )

    # 6. Find highest-scoring products
    top_indices = candidate_indices[
        np.argsort(scores[candidate_indices])[::-1][:top_n]
    ]

    # 7. Create recommendation DataFrame
    recommendations = pd.DataFrame({
        "PRODUCT_ID": cf_product_ids.take(top_indices),
        "recommendation_score": scores[top_indices],
    })

    # 8. Add product descriptions
    recommendations = recommendations.merge(
        valid_catalog[
            ["PRODUCT_ID", "product_label"]
        ],
        on="PRODUCT_ID",
        how="left",
        validate="one_to_one",
    )

    return recommendations[
        ["PRODUCT_ID", "product_label", "recommendation_score"]
    ].reset_index(drop=True)

In [90]:

# Select the same customer
CUSTOMER_ID = 1

# Generate SVD recommendations
svd_recommendations = recommend_svd(
    household_key=CUSTOMER_ID,
    top_n=10,
)

print("Selected Customer:", CUSTOMER_ID)

print(
    "Previously Purchased Products:",
    customer_item_matrix[
        customer_ids.get_loc(CUSTOMER_ID)
    ].nnz
)

display(svd_recommendations.round(4))


Selected Customer: 1
Previously Purchased Products: 400


,PRODUCT_ID,product_label,recommendation_score
0,981760,EGGS - X-LARGE | 1 DZ | ID: 981760,0.649
1,962568,CELERY | ID: 962568,0.631
2,1127831,STRAWBERRIES | 16 OZ | ID: 1127831,0.619
3,878996,GRAPES RED | 18 LB | ID: 878996,0.587
4,1024306,CANTALOUPE WHOLE | ID: 1024306,0.548
5,854852,TOMATOES HOTHOUSE ON THE VINE | 13 LB | ID: 85...,0.527
6,1027569,CARDS EVERYDAY | ID: 1027569,0.524
7,1022254,POTATOES SWEET&YAMS | 40 LB | ID: 1022254,0.513
8,911812,CONDENSED SOUP | 10.5OZ | ID: 911812,0.483
9,1050229,CONDENSED SOUP | 10.7OZ | ID: 1050229,0.468



### 6.5.3 Non-negative Matrix Factorization (NMF)

NMF is a matrix factorization algorithm.

It decomposes a non-negative matrix into two smaller non-negative matrices.

We will use our binary customer-product interaction matrix.

Our goals are to:

- Learn customer and product latent factors.
- Calculate predicted interaction scores.
- Generate product recommendations.
- Compare NMF with Truncated SVD.

We will start with 50 latent components.


In [93]:
# 1. Define model parameters
NMF_COMPONENTS = 50

# 2. Validate input matrix
assert (customer_item_matrix.data >= 0).all()

# 3. Initialize NMF model
nmf_model = NMF(
    n_components=NMF_COMPONENTS,
    init="nndsvda",
    solver="cd",
    max_iter=800,
    tol=1e-4,
    random_state=42,
)

# 4. Train NMF model
nmf_user_factors = nmf_model.fit_transform(
    customer_item_matrix
)

# 5. Extract product factors
nmf_item_factors = nmf_model.components_.T

# 6. Validate results
assert nmf_user_factors.shape == (
    len(customer_ids),
    NMF_COMPONENTS,
)

assert nmf_item_factors.shape == (
    len(cf_product_ids),
    NMF_COMPONENTS,
)

assert (nmf_user_factors >= 0).all()
assert (nmf_item_factors >= 0).all()

# 7. Display results
print("Original Matrix:", customer_item_matrix.shape)
print("NMF User Factors:", nmf_user_factors.shape)
print("NMF Item Factors:", nmf_item_factors.shape)

print("Iterations:", nmf_model.n_iter_)
print(
    "Reconstruction Error:",
    round(nmf_model.reconstruction_err_, 4)
)

Original Matrix: (2481, 11603)
NMF User Factors: (2481, 50)
NMF Item Factors: (11603, 50)
Iterations: 800
Reconstruction Error: 855.6686


c:\Users\okand\Desktop\Projects\dunnhumby-retail-intelligence\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 800 reached. Increase it to improve convergence.
  warnings.warn(



### 6.5.4 NMF-Based Product Recommendations

We will generate product recommendations using NMF latent factors.

The function will:

- Find the selected customer.
- Calculate product interaction scores.
- Exclude previously purchased products.
- Rank products by their scores.
- Return the top recommendations.

We will compare NMF recommendations with SVD and User-Based Collaborative Filtering.


In [94]:

def recommend_nmf(
    household_key: int,
    top_n: int = 10,
) -> pd.DataFrame:
    """Recommend products using NMF latent factors."""

    if top_n < 1:
        raise ValueError("top_n must be positive.")

    # 1. Find customer index
    customer_idx = customer_ids.get_indexer(
        [household_key]
    )[0]

    if customer_idx == -1:
        raise ValueError("Customer is not available in the model.")

    # 2. Calculate predicted product scores
    scores = np.asarray(
        nmf_user_factors[customer_idx]
        @ nmf_item_factors.T
    ).ravel()

    # 3. Find previously purchased products
    purchased_indices = customer_item_matrix[
        customer_idx
    ].indices

    # 4. Exclude purchased products
    scores[purchased_indices] = -np.inf

    # 5. Keep valid positive scores
    candidate_indices = np.flatnonzero(
        np.isfinite(scores) & (scores > 0)
    )

    # 6. Select highest-scoring products
    top_indices = candidate_indices[
        np.argsort(scores[candidate_indices])[::-1][:top_n]
    ]

    # 7. Create recommendation results
    recommendations = pd.DataFrame({
        "PRODUCT_ID": cf_product_ids.take(top_indices),
        "recommendation_score": scores[top_indices],
    })

    # 8. Add product descriptions
    recommendations = recommendations.merge(
        valid_catalog[
            ["PRODUCT_ID", "product_label"]
        ],
        on="PRODUCT_ID",
        how="left",
        validate="one_to_one",
    )

    return recommendations[
        [
            "PRODUCT_ID",
            "product_label",
            "recommendation_score",
        ]
    ].reset_index(drop=True)


In [95]:

# Select customer
CUSTOMER_ID = 1

# Generate NMF recommendations
nmf_recommendations = recommend_nmf(
    household_key=CUSTOMER_ID,
    top_n=10,
)

print("Selected Customer:", CUSTOMER_ID)

print(
    "Previously Purchased Products:",
    customer_item_matrix[
        customer_ids.get_loc(CUSTOMER_ID)
    ].nnz
)

display(nmf_recommendations.round(4))


Selected Customer: 1
Previously Purchased Products: 400


,PRODUCT_ID,product_label,recommendation_score
0,1027569,CARDS EVERYDAY | ID: 1027569,0.687
1,1127831,STRAWBERRIES | 16 OZ | ID: 1127831,0.660
2,981760,EGGS - X-LARGE | 1 DZ | ID: 981760,0.590
3,878996,GRAPES RED | 18 LB | ID: 878996,0.585
4,866211,GRAPES WHITE | 18 LB | ID: 866211,0.567
5,1022097,CARDS EVERYDAY | ID: 1022097,0.561
6,852856,CARDS EVERYDAY | ID: 852856,0.542
7,912704,CARDS EVERYDAY | ID: 912704,0.514
8,962568,CELERY | ID: 962568,0.474
9,1024306,CANTALOUPE WHOLE | ID: 1024306,0.469



### 6.5.5 Save Matrix Factorization Models

We will save the trained SVD and NMF models.

We will also save customer and product latent factors.

These artifacts will allow us to generate recommendations without retraining the models.

The NMF model reached the maximum iteration limit, so its convergence is not confirmed.


In [97]:
# 1. Create model directory
model_dir = PROJECT_ROOT / "models" / "recommendation"
model_dir.mkdir(parents=True, exist_ok=True)

# 2. Save trained models
joblib.dump(
    svd_model,
    model_dir / "svd_model.joblib"
)

joblib.dump(
    nmf_model,
    model_dir / "nmf_model.joblib"
)

# 3. Save latent factors and ID mappings
np.savez_compressed(
    model_dir / "matrix_factorization_factors.npz",
    svd_user_factors=user_factors,
    svd_item_factors=item_factors,
    nmf_user_factors=nmf_user_factors,
    nmf_item_factors=nmf_item_factors,
    customer_ids=customer_ids.to_numpy(),
    product_ids=cf_product_ids.to_numpy(),
)

# 4. Save model metadata
factorization_metadata = {
    "svd_components": svd_model.n_components,
    "nmf_components": nmf_model.n_components,
    "nmf_iterations": nmf_model.n_iter_,
    "nmf_converged": (
        nmf_model.n_iter_ < nmf_model.max_iter
    ),
    "training_end_day": TRAIN_END_DAY,
}

joblib.dump(
    factorization_metadata,
    model_dir / "factorization_metadata.joblib"
)

print("Matrix Factorization models saved successfully.")

for path in sorted(model_dir.glob("*factor*")):
    print(path.name)

Matrix Factorization models saved successfully.
factorization_metadata.joblib
matrix_factorization_factors.npz



## 6.6 Model Evaluation

### 6.6.1 Evaluation Dataset Preparation

We will evaluate recommendation models using future purchases.

Training period: Days 1–621

Test period: Days 622–711

We will focus on new product discovery.

Our evaluation will include customers available in the training data and products supported by the collaborative filtering models.

Products already purchased during training will not be counted as new products.

We will use this dataset to calculate ranking metrics.


In [99]:
cf_interactions.head()

,household_key,PRODUCT_ID
0,1,825123
1,1,831447
2,1,840361
3,1,845307
4,1,852014


In [100]:
# 1. Get unique customer-product pairs from test data
test_pairs = test_items[
    ["household_key", "PRODUCT_ID"]
].drop_duplicates()

# 2. Keep known customers and eligible products
eligible_test = test_pairs.loc[
    test_pairs["household_key"].isin(customer_ids)
    & test_pairs["PRODUCT_ID"].isin(cf_product_ids)
].copy()

# 3. Get historical customer-product interactions
train_pairs = cf_interactions[
    ["household_key", "PRODUCT_ID"]
].drop_duplicates()

# 4. Identify products already purchased in training
evaluation_pairs = eligible_test.merge(
    train_pairs.assign(purchased_before=1),
    on=["household_key", "PRODUCT_ID"],
    how="left",
    validate="one_to_one",
)

# 5. Keep new products only
novel_test_items = evaluation_pairs.loc[
    evaluation_pairs["purchased_before"].isna(),
    ["household_key", "PRODUCT_ID"]
].copy()

# 6. Create actual product sets for each customer
actual_products = (
    novel_test_items
    .groupby("household_key")["PRODUCT_ID"]
    .agg(set)
    .to_dict()
)

# 7. Select customers with at least one eligible new product
evaluation_customers = pd.Index(
    sorted(actual_products.keys())
)

# 8. Display evaluation summary
print("Test Customer-Product Pairs:", len(test_pairs))
print("Eligible Test Pairs:", len(eligible_test))
print("Novel Test Pairs:", len(novel_test_items))
print("Evaluation Customers:", len(evaluation_customers))

print(
    "Customer Coverage (%):",
    round(
        len(evaluation_customers) / len(customer_ids) * 100,
        2
    )
)

display(novel_test_items.head(10))

Test Customer-Product Pairs: 275481
Eligible Test Pairs: 207277
Novel Test Pairs: 109658
Evaluation Customers: 2266
Customer Coverage (%): 91.33


,household_key,PRODUCT_ID
2,1,829563
3,1,830156
8,1,901062
9,1,904959
16,1,1138467
22,1,821867
23,1,834484
26,1,907957
28,1,951954
30,1,962568


* Of the 207,277 valid test associations, 109,658 consist of products that the customer did not purchase during the training period.


### 6.6.2 Recommendation Evaluation Metrics

We will create a reusable evaluation function.

We will calculate:

- Precision@K
- Recall@K
- Hit Rate@K
- Catalog Coverage@K

All models will use the same evaluation customers and product catalog.

We will compare the recommendation models with a popularity-based baseline.

Notes:

- Precision@10: How many of the 10 recommended products were actually purchased.
- Recall@10: The proportion of newly purchased products that were actually recommended.
- Hit Rate@10: The proportion of customers who were provided with at least one correct recommendation.
- Coverage@10: The extent to which the product catalog is included in recommendations.



In [101]:
def evaluate_recommender(
    recommend_fn,
    customers,
    actual_products,
    k=10,
    catalog_size=None,
):
    """Evaluate top-K recommendations against future purchases."""

    if k < 1:
        raise ValueError("k must be positive.")

    if catalog_size is None:
        catalog_size = len(cf_product_ids)

    results = []
    all_recommended = set()

    for household_key in customers:

        # 1. Generate recommendations
        recommendations = recommend_fn(
            household_key=int(household_key),
            top_n=k,
        )

        recommended_ids = (
            recommendations["PRODUCT_ID"]
            .drop_duplicates()
            .head(k)
            .tolist()
        )

        # 2. Get actual future purchases
        actual_ids = actual_products[int(household_key)]

        # 3. Find correct recommendations
        hits = len(set(recommended_ids) & actual_ids)

        # 4. Calculate ranking metrics
        precision = hits / k
        recall = hits / len(actual_ids)
        hit_rate = int(hits > 0)

        all_recommended.update(recommended_ids)

        results.append({
            "household_key": household_key,
            "precision": precision,
            "recall": recall,
            "hit_rate": hit_rate,
        })

    # 5. Calculate average performance
    results_df = pd.DataFrame(results)

    summary = {
        "precision_at_k": results_df["precision"].mean(),
        "recall_at_k": results_df["recall"].mean(),
        "hit_rate_at_k": results_df["hit_rate"].mean(),
        "coverage_at_k": (
            len(all_recommended) / catalog_size
        ),
    }

    return summary, results_df

In [102]:
# 1. Rank products by number of unique purchasing customers
popular_product_ids = (
    product_customer_counts
    .loc[cf_product_ids]
    .sort_values(ascending=False)
    .index
    .tolist()
)

def recommend_popular(
    household_key: int,
    top_n: int = 10,
) -> pd.DataFrame:
    """Recommend popular products not previously purchased."""

    if top_n < 1:
        raise ValueError("top_n must be positive.")

    customer_idx = customer_ids.get_indexer(
        [household_key]
    )[0]

    if customer_idx == -1:
        raise ValueError("Unknown customer.")

    # Find previously purchased products
    purchased_ids = set(
        cf_product_ids.take(
            customer_item_matrix[customer_idx].indices
        )
    )

    # Select popular products not previously purchased
    recommendations = []

    for product_id in popular_product_ids:

        if product_id not in purchased_ids:
            recommendations.append(product_id)

        if len(recommendations) == top_n:
            break

    return pd.DataFrame({
        "PRODUCT_ID": recommendations
    })

In [103]:
# 2. Evaluate popularity baseline
popularity_summary, popularity_details = evaluate_recommender(
    recommend_fn=recommend_popular,
    customers=evaluation_customers,
    actual_products=actual_products,
    k=10,
)

# 3. Display results
popularity_results = pd.DataFrame(
    [popularity_summary],
    index=["Popularity Baseline"],
)

display(popularity_results.round(4))

,precision_at_k,recall_at_k,hit_rate_at_k,coverage_at_k
Popularity Baseline,0.068,0.017,0.439,0.007



### 6.6.4 Collaborative Filtering and Matrix Factorization Evaluation

We will evaluate three recommendation models:

- User-Based Collaborative Filtering
- Truncated SVD
- Non-negative Matrix Factorization

All models will use the same evaluation customers and future purchase data.

We will compare Precision@10, Recall@10, Hit Rate@10, and Catalog Coverage@10.

The Popularity Baseline will be our reference model.


In [104]:
# Define recommendation models
models_to_evaluate = {
    "User-Based CF": recommend_user_based,
    "Truncated SVD": recommend_svd,
    "NMF": recommend_nmf,
}

# Store evaluation results
model_results = []
model_details = {}

for model_name, recommend_fn in models_to_evaluate.items():

    print(f"Evaluating: {model_name}")

    # Evaluate the recommendation model
    summary, details = evaluate_recommender(
        recommend_fn=recommend_fn,
        customers=evaluation_customers,
        actual_products=actual_products,
        k=10,
        catalog_size=len(cf_product_ids),
    )

    # Store summary metrics
    model_results.append({
        "model": model_name,
        **summary,
    })

    # Store customer-level results
    model_details[model_name] = details

# Create evaluation DataFrame
model_evaluation = (
    pd.DataFrame(model_results)
    .set_index("model")
)

# Compare with Popularity Baseline
model_comparison = pd.concat([
    popularity_results,
    model_evaluation,
])

# Sort by Recall@10
model_comparison = model_comparison.sort_values(
    "recall_at_k",
    ascending=False,
)

print("\nRECOMMENDATION MODEL COMPARISON")

display(model_comparison.round(4))

Evaluating: User-Based CF
Evaluating: Truncated SVD
Evaluating: NMF

RECOMMENDATION MODEL COMPARISON


,precision_at_k,recall_at_k,hit_rate_at_k,coverage_at_k
Truncated SVD,0.082,0.022,0.516,0.062
User-Based CF,0.081,0.020,0.508,0.029
NMF,0.081,0.020,0.509,0.065
Popularity Baseline,0.068,0.017,0.439,0.007



### 6.6.5 Association Rules Evaluation

We will adapt Association Rules to generate customer-level recommendations.

The function will:

- Identify products purchased by a customer.
- Find matching association rules.
- Exclude previously purchased products.
- Rank recommendations by Confidence and Lift.
- Return the top K products.

We will evaluate the model using the same customers and test period.


In [105]:
# 1. Create historical purchase sets
train_purchase_sets = (
    train_items
    .groupby("household_key")["PRODUCT_ID"]
    .agg(set)
    .to_dict()
)

# 2. Keep rules for products in the evaluation catalog
cf_product_set = set(cf_product_ids)

evaluation_rules = recommendation_rules.loc[
    recommendation_rules["consequent_id"].isin(cf_product_set)
].copy()

In [106]:
def recommend_association_customer(
    household_key: int,
    top_n: int = 10,
) -> pd.DataFrame:
    """Recommend new products using customer purchase history and association rules."""

    if top_n < 1:
        raise ValueError("top_n must be positive.")

    if household_key not in train_purchase_sets:
        raise ValueError("Customer is not available.")

    # 3. Get historical purchases
    purchased_ids = train_purchase_sets[household_key]

    # 4. Find rules matching purchased products
    candidates = evaluation_rules.loc[
        evaluation_rules["antecedent_id"].isin(purchased_ids)
        & ~evaluation_rules["consequent_id"].isin(purchased_ids)
    ].copy()

     # 5. Rank candidate products
    candidates = (
        candidates
        .sort_values(
            ["confidence", "lift", "support"],
            ascending=False,
        )
        .drop_duplicates("consequent_id")
        .head(top_n)
    )

    # 6. Prepare recommendation results
    recommendations = candidates.rename(
        columns={
            "consequent_id": "PRODUCT_ID",
            "confidence": "recommendation_score",
        }
    )[
        ["PRODUCT_ID", "recommendation_score", "lift", "support"]
    ].copy()

    return recommendations.reset_index(drop=True)


In [107]:
# 7. Evaluate Association Rules
association_summary, association_details = evaluate_recommender(
    recommend_fn=recommend_association_customer,
    customers=evaluation_customers,
    actual_products=actual_products,
    k=10,
    catalog_size=len(cf_product_ids),
)

association_evaluation = pd.DataFrame(
    [association_summary],
    index=["Association Rules"],
)

display(association_evaluation.round(4))

,precision_at_k,recall_at_k,hit_rate_at_k,coverage_at_k
Association Rules,0.037,0.010,0.289,0.002



### 6.6.6 Item-Based Collaborative Filtering Evaluation

We will create customer-level recommendations using Item-Based Collaborative Filtering.

For each customer, we will:

1. Select the 10 most recently purchased unique products.
2. Find similar products using Item-Based KNN.
3. Combine similarity scores.
4. Exclude previously purchased products.
5. Recommend the top 10 products.

We will evaluate the recommendations using future purchase data.


In [109]:
# Define recommendation parameters
N_SEED_ITEMS = 10
N_ITEM_NEIGHBORS = 30

# Keep products supported by the Item-Based model
historical_items = train_items.loc[
    train_items["PRODUCT_ID"].isin(cf_product_ids),
    ["household_key", "PRODUCT_ID", "basket_day"]
]

# Find each customer's last purchase day for each product
last_product_purchases = (
    historical_items
    .groupby(
        ["household_key", "PRODUCT_ID"],
        as_index=False
    )
    .agg(last_day=("basket_day", "max"))
)

# Sort purchases from most recent to oldest
last_product_purchases = last_product_purchases.sort_values(
    ["household_key", "last_day", "PRODUCT_ID"],
    ascending=[True, False, True],
)

# Select the 10 most recent unique products per customer
recent_products = (
    last_product_purchases
    .groupby("household_key")
    .head(N_SEED_ITEMS)
)

# Create customer-product dictionary
customer_seed_products = (
    recent_products
    .groupby("household_key")["PRODUCT_ID"]
    .agg(list)
    .to_dict()
)

print("Customers With Seed Products:", len(customer_seed_products))

print(
    "Customer 1 Seed Products:",
    customer_seed_products.get(1, [])
)

Customers With Seed Products: 2481
Customer 1 Seed Products: [829323, 851515, 879280, 940947, 968164, 979792, 995242, 1055201, 1069575, 1074612]


* Instead of recalculating the nearest neighbors for the same product for every customer, we will find them once and store them in memory.

In [110]:
@lru_cache(maxsize=10000)
def get_item_neighbors(product_id: int) -> tuple:
    """Return cached nearest products and similarity scores."""

    product_idx = cf_product_ids.get_indexer(
        [product_id]
    )[0]

    distances, indices = item_knn.kneighbors(
        item_user_matrix[product_idx],
        n_neighbors=min(
            N_ITEM_NEIGHBORS + 1,
            len(cf_product_ids),
        ),
    )

    neighbors = []

    for idx, distance in zip(indices[0], distances[0]):
        neighbor_id = int(cf_product_ids[idx])
        similarity = float(1 - distance)

        if neighbor_id != product_id and similarity > 0:
            neighbors.append((neighbor_id, similarity))

    return tuple(neighbors)

In [111]:
def recommend_item_based_customer(
    household_key: int,
    top_n: int = 10,
) -> pd.DataFrame:
    """Recommend new products using Item-Based KNN."""

    if top_n < 1:
        raise ValueError("top_n must be positive.")

    customer_idx = customer_ids.get_indexer(
        [household_key]
    )[0]

    if customer_idx == -1:
        raise ValueError("Unknown customer.")

    # 1. Get recent products
    seed_products = customer_seed_products.get(
        household_key, []
    )

    # 2. Get all previously purchased products
    purchased_ids = set(
        cf_product_ids.take(
            customer_item_matrix[customer_idx].indices
        )
    )

    # 3. Combine product similarity scores
    scores = defaultdict(float)

    for product_id in seed_products:

        neighbors = get_item_neighbors(int(product_id))

        for neighbor_id, similarity in neighbors:

            if neighbor_id not in purchased_ids:
                scores[neighbor_id] += similarity

    # 4. Rank candidate products
    top_products = sorted(
        scores.items(),
        key=lambda item: (-item[1], item[0]),
    )[:top_n]

    return pd.DataFrame(
        top_products,
        columns=["PRODUCT_ID", "recommendation_score"],
    )

In [113]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*sklearn\.utils\.parallel\.delayed.*",
    category=UserWarning,
)


# Test recommendation function
example_recommendations = recommend_item_based_customer(
    household_key=1,
    top_n=10,
)

# Add product descriptions for inspection
display(
    example_recommendations.merge(
        valid_catalog[["PRODUCT_ID", "product_label"]],
        on="PRODUCT_ID",
        how="left",
    ).round(4)
)

# Evaluate Item-Based CF
item_based_summary, item_based_details = evaluate_recommender(
    recommend_fn=recommend_item_based_customer,
    customers=evaluation_customers,
    actual_products=actual_products,
    k=10,
    catalog_size=len(cf_product_ids),
)

item_based_evaluation = pd.DataFrame(
    [item_based_summary],
    index=["Item-Based CF"],
)

display(item_based_evaluation.round(4))

print("\nNeighbor Cache:")
print(get_item_neighbors.cache_info())


,PRODUCT_ID,recommendation_score,product_label
0,1127831,2.045,STRAWBERRIES | 16 OZ | ID: 1127831
1,981760,1.823,EGGS - X-LARGE | 1 DZ | ID: 981760
2,866211,1.656,GRAPES WHITE | 18 LB | ID: 866211
3,878996,1.651,GRAPES RED | 18 LB | ID: 878996
4,854852,1.611,TOMATOES HOTHOUSE ON THE VINE | 13 LB | ID: 85...
5,1098066,1.448,HOT DOG BUNS | 11 OZ | ID: 1098066
6,826249,1.404,HAMBURGER BUNS | 12 OZ | ID: 826249
7,916122,1.340,CHICKEN BREAST BONELESS | ID: 916122
8,844179,1.270,PRIMAL | ID: 844179
9,1044078,1.243,LEAN | ID: 1044078


,precision_at_k,recall_at_k,hit_rate_at_k,coverage_at_k
Item-Based CF,0.073,0.019,0.471,0.086



Neighbor Cache:
CacheInfo(hits=38553, misses=6637, maxsize=10000, currsize=6637)



### 6.6.7 Content-Based Filtering Evaluation

We will evaluate Content-Based Filtering using customer purchase histories.

For each customer, we will:

1. Select the 10 most recently purchased unique products.
2. Find similar products using TF-IDF and categorical features.
3. Combine similarity scores.
4. Remove previously purchased products.
5. Return the top 10 recommendations.

We will compare the results with other recommendation models.


In [114]:
# Use a single worker to avoid repeated parallel warnings
content_knn.set_params(n_jobs=1)

# Verify product ID alignment
assert np.array_equal(
    content_product_ids.to_numpy(),
    cf_product_ids.to_numpy()
)

# Cache similar products
@lru_cache(maxsize=12000)
def get_content_neighbors(product_id: int) -> tuple:
    """Return similar products using content features."""

    # Find product position
    product_idx = content_product_ids.get_indexer(
        [product_id]
    )[0]

    if product_idx == -1:
        return tuple()

    # Find nearest products
    distances, indices = content_knn.kneighbors(
        content_matrix[product_idx],
        n_neighbors=min(
            N_ITEM_NEIGHBORS + 1,
            len(content_product_ids)
        ),
    )

    neighbors = []

    for idx, distance in zip(indices[0], distances[0]):

        neighbor_id = int(content_product_ids[idx])
        similarity = float(1 - distance)

        if neighbor_id != product_id and similarity > 0:
            neighbors.append((neighbor_id, similarity))

    return tuple(neighbors)


In [115]:

def recommend_content_customer(
    household_key: int,
    top_n: int = 10,
) -> pd.DataFrame:
    """Recommend products using customer history and content similarity."""

    if top_n < 1:
        raise ValueError("top_n must be positive.")

    # 1. Find customer
    customer_idx = customer_ids.get_indexer(
        [household_key]
    )[0]

    if customer_idx == -1:
        raise ValueError("Unknown customer.")

    # 2. Get recent purchased products
    seed_products = customer_seed_products.get(
        household_key, []
    )

    # 3. Get all historical purchases
    purchased_ids = set(
        cf_product_ids.take(
            customer_item_matrix[customer_idx].indices
        )
    )

    # 4. Calculate recommendation scores
    scores = defaultdict(float)

    for product_id in seed_products:

        neighbors = get_content_neighbors(int(product_id))

        for neighbor_id, similarity in neighbors:

            if neighbor_id not in purchased_ids:
                scores[neighbor_id] += similarity

    # 5. Rank products
    top_products = sorted(
        scores.items(),
        key=lambda item: (-item[1], item[0])
    )[:top_n]

    return pd.DataFrame(
        top_products,
        columns=["PRODUCT_ID", "recommendation_score"]
    )


In [116]:

# 1. Generate recommendations for Customer 1
content_customer_recommendations = recommend_content_customer(
    household_key=1,
    top_n=10,
)

# 2. Add product descriptions
display(
    content_customer_recommendations.merge(
        valid_catalog[["PRODUCT_ID", "product_label"]],
        on="PRODUCT_ID",
        how="left",
        validate="one_to_one",
    ).round(4)
)

# 3. Evaluate Content-Based Filtering
content_summary, content_details = evaluate_recommender(
    recommend_fn=recommend_content_customer,
    customers=evaluation_customers,
    actual_products=actual_products,
    k=10,
    catalog_size=len(cf_product_ids),
)

# 4. Display results
content_evaluation = pd.DataFrame(
    [content_summary],
    index=["Content-Based Filtering"],
)

display(content_evaluation.round(4))

print("\nCache Information:")
print(get_content_neighbors.cache_info())


,PRODUCT_ID,recommendation_score,product_label
0,824073,1.000,FLUID MILK WHITE ONLY | ID: 824073
1,826597,1.000,ENTREES | 24 OZ | ID: 826597
2,827261,1.000,PEACHES | 15 OZ | ID: 827261
3,835715,1.000,PEACHES | 15 OZ | ID: 835715
4,837339,1.000,BEANS GREEN: FS/WHL/CUT | 14.5 OZ | ID: 837339
5,837669,1.000,CREAM CHEESE | 8 OZ | ID: 837669
6,843346,1.000,CREAM CHEESE | 8 OZ | ID: 843346
7,843756,1.000,CREAM CHEESE | 8 OZ | ID: 843756
8,852864,1.000,BEANS GREEN: FS/WHL/CUT | 14.5 OZ | ID: 852864
9,856435,1.000,CREAM CHEESE | 8 OZ | ID: 856435


,precision_at_k,recall_at_k,hit_rate_at_k,coverage_at_k
Content-Based Filtering,0.019,0.005,0.154,0.493



Cache Information:
CacheInfo(hits=15958, misses=6637, maxsize=12000, currsize=6637)



### 6.6.8 Final Recommendation Model Comparison

We evaluated six recommendation approaches and a popularity baseline.

All models were evaluated using historical training data and future customer purchases.

We compared Precision@10, Recall@10, Hit Rate@10, and Catalog Coverage@10.

Truncated SVD achieved the highest ranking performance in this experiment.

Content-Based Filtering achieved the highest catalog coverage but the lowest recommendation accuracy.

These results are exploratory because the test period was used to compare model choices.


In [117]:

# Combine all evaluation results
final_comparison = pd.concat([
    popularity_results,
    association_evaluation,
    model_evaluation,
    item_based_evaluation,
    content_evaluation,
])

# Sort by Precision@10
final_comparison = final_comparison.sort_values(
    "precision_at_k",
    ascending=False,
)

# Rename columns for readability
final_comparison.index.name = "model"

print("FINAL RECOMMENDATION MODEL COMPARISON")

display(final_comparison.round(4))

# Save evaluation results
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

final_comparison.to_csv(
    output_dir / "recommendation_model_comparison.csv"
)

print("Model comparison saved successfully.")


FINAL RECOMMENDATION MODEL COMPARISON


,precision_at_k,recall_at_k,hit_rate_at_k,coverage_at_k
model,,,,
Truncated SVD,0.082,0.022,0.516,0.062
NMF,0.081,0.020,0.509,0.065
User-Based CF,0.081,0.020,0.508,0.029
Item-Based CF,0.073,0.019,0.471,0.086
Popularity Baseline,0.068,0.017,0.439,0.007
Association Rules,0.037,0.010,0.289,0.002
Content-Based Filtering,0.019,0.005,0.154,0.493


Model comparison saved successfully.



## 6.7 Key Findings and Conclusions

### Recommendation Models

We developed six recommendation approaches:

1. Association Rule Learning
2. Item-Based Collaborative Filtering
3. User-Based Collaborative Filtering
4. Content-Based Filtering
5. Truncated SVD
6. Non-negative Matrix Factorization

We also created a Popularity Baseline.

### Evaluation Results

Truncated SVD achieved the best Precision@10 (0.082) and Hit Rate@10 (0.516).

NMF and User-Based Collaborative Filtering achieved similar performance.

Item-Based Collaborative Filtering performed better than the Popularity Baseline.

Association Rules showed limited performance for personalized new-product discovery.

Content-Based Filtering achieved high catalog coverage but low recommendation accuracy.

### Limitations

- The evaluation focused on new product discovery.
- Previously purchased products were excluded.
- Some customers and products were not supported by the models.
- The product catalog contains many similar product descriptions.
- NMF did not fully converge.
- Only one future test period was evaluated.
- Model selection using the test results requires further independent validation.

### Conclusion

Truncated SVD is the strongest candidate for personalized new-product recommendations in our current experiments.

Association Rules and Content-Based Filtering may support different recommendation scenarios.

Future work should focus on independent validation, model integration, and deployment.
